# Setup

In [1]:
from pyiron import Project
import os
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display
from datetime import datetime
from ase.eos import EquationOfState

In [2]:
# Global Variables
VaspExecutable = "6.4.3_std_mpi"

ConversionFactor_eV_A3toGPa = 160.218

# Useful Modules

In [3]:
def CreateStrainRange(ZeroStrainPoint_Percent:float = 0.0, Range_Percent:float = 0.1, NumberOfStrains: int = 5, Regime:str = "even"):
    StrainRange = []
    if Regime == "even":
        # Ensure that the strain distribution is centered around 0% strain
        if NumberOfStrains % 2 == 0:
            NumberOfStrains += 1
        
        StepSize = Range_Percent/int(NumberOfStrains/2)
        OneSidedStrain = [Strain.item() for Strain in np.linspace(ZeroStrainPoint_Percent+StepSize, Range_Percent, num=int(NumberOfStrains/2))]
        
        for Strain in reversed(OneSidedStrain):
            StrainRange.append(-1*Strain)
        StrainRange.append(ZeroStrainPoint_Percent)
        for Strain in OneSidedStrain:
            StrainRange.append(Strain)
    return StrainRange

In [4]:
def ConvertToDirectoryName(Value):
    # Convert float numbers to strings
    # - -> n; + -> p, . -> d
    # e.g. -0.5 ==> n0d5
    Sign = "n" if Value < 0 else "p"
    if isinstance(Value, int):
        ReturnString = f"{Sign}{abs(Value)}d0"
    else:
        ReturnString = f"{Sign}{abs(Value)}".replace(".", "d")
    return ReturnString

def ParseDirectoryName(DirectoryName: str):
    Sign = -1 if DirectoryName[0] == "n" else 1
    NumberStrings = DirectoryName.split("d")
    OutputNumber = float(NumberStrings[0][1:] + "." + NumberStrings[1])
    return Sign * OutputNumber

In [5]:
def WaitForJobs(Project):
    # Wait for jobs to be finished
    while not all(Project.job_table()["status"].isin(["finished", "aborted"])):
        clear_output(wait=True)
        display(Project.job_table())
        time.sleep(5)
    
    clear_output(wait=True)
    display(Project.job_table())

In [6]:
# Creates a log file, to dump info during runtime
class LoggingFile():
    def __init__(self, FilePath: str, Mode: str = "w"):
        self.FilePath = FilePath
        self.File = open(FilePath, Mode)

    def WriteInformation(self, Input: str, WriteToNewParagraph: bool = False, InsertLineBreak: bool = True, InsertTimeStamp: bool = False):
        LoggingMessage = ""
        if WriteToNewParagraph:
            LoggingMessage += "         |\n"
        if InsertTimeStamp:
            LoggingMessage += f"{datetime.now():%H:%M:%S} | "
        else:
            LoggingMessage += "         | "
        LoggingMessage += Input
        if InsertLineBreak:
            LoggingMessage += "\n"

        self.File.write(str(LoggingMessage))
        self.File.flush()

    def Close(self):
        self.File.close()

    def __del__(self):
        if not self.File.closed:
            self.File.close()

# Calculations
## Porject definition

In [7]:
SimulationFilePath = "SimulationFileDump"

In [8]:
if "Log" in globals():
    Log.Close()
Log = LoggingFile("Outputs/ElasticConstants.log")

Log.WriteInformation("Start bulk modulus calculation", InsertTimeStamp=True)

ElasticConstantsProject = Project(path=SimulationFilePath)
ElasticConstantsProject.remove_jobs(recursive=True, silently=True)

Project(): 0.0 s
Empty DataFrame
Columns: [job, status]
Index: []
job_table(): 0.0 s


0it [00:00, ?it/s]

In [9]:
UnitCell = ElasticConstantsProject.create.structure.ase.bulk("Al", cubic=True, a = 4.04)
#UnitCell.plot3d()

## Initial Relaxation

To apply accurate strains, the structure needs to be in the minimum energy state. This is achieved by first relaxing the structure with VASP, reading the resulting lattice constant and the manually searching for the minimum, by applying small uniform strains

In [ ]:
# Create relaxation Job
InitialRelaxationJob = ElasticConstantsProject.create_job(job_type=ElasticConstantsProject.job_type.Vasp, job_name="InitialRelaxation", delete_existing_job=True)
InitialRelaxationJob.server.queue = "small"
InitialRelaxationJob.server.cores = 4

InitialRelaxationJob.structure = UnitCell.copy()

InitialRelaxationJob.input.incar["ENCUT"] = 650
InitialRelaxationJob.input.incar["IBRION"] = 2
InitialRelaxationJob.input.incar["ISIF"] = 3
InitialRelaxationJob.input.incar["NSW"] = 200

Log.WriteInformation(str(InitialRelaxationJob.input.incar))

InitialRelaxationJob.executable = VaspExecutable
InitialRelaxationJob.run()
WaitForJobs(ElasticConstantsProject)

RelaxedStructure = InitialRelaxationJob.structure.copy()
RelaxedStructure.set_cell(InitialRelaxationJob.output.cells[-1], scale_atoms=False)
RelaxedStructure.positions = InitialRelaxationJob.output.positions[-1]
ActiveVolume_A3 = RelaxedStructure.get_volume()

,id,status,chemicalformula,job,subjob,projectpath,project,timestart,timestop,totalcputime,computer,hamilton,hamversion,parentid,masterid
0,39,submitted,Al4,InitialRelaxation,/InitialRelaxation,None,/scratch/htc/jhornig/01_PyIron/PyIronWorkflows/src/SimulationFileDump/,2026-11-23 11:29:24.872232,None,None,pyiron@htc-login03.zib.de#4#small,Vasp,6.4.3_std_mpi,None,None


In [ ]:
def SetVolume(Structure, TargetVolume_A3):
    NewStructure = Structure.copy()
    Scaling      = (TargetVolume_A3 / Structure.get_volume()) ** (1/3)
    NewStructure.set_cell(Structure.cell * Scaling, scale_atoms=True)
    return NewStructure, Scaling

In [ ]:
# Creates a strained state using the input strain in % as linear strain, applied in all directions equally.
def RunUniaxialStrains(Log, MaximumStrainMagnitude_Percent, UnstrainedStructure, Iteration):    
    StrainMagnitudes_Percent = CreateStrainRange(ZeroStrainPoint_Percent=0, Range_Percent=MaximumStrainMagnitude_Percent, NumberOfStrains=7)
    Log.WriteInformation(f"Strain Magnitudes [  %]: [{', '.join(f'{Data:.3f}' for Data in StrainMagnitudes_Percent)}]")
    for Strain_Percent in StrainMagnitudes_Percent:
        StrainFactor     = 1 + Strain_Percent / 100
        TargetVolume_A3  = UnstrainedStructure.get_volume() * StrainFactor ** 3

        ActiveStructure, _ = SetVolume(UnstrainedStructure, TargetVolume_A3)

        JobName = f"StructureRelaxation_Iter{Iteration}_{ConvertToDirectoryName(round(Strain_Percent, 4))}"
        RelaxationJob = ElasticConstantsProject.create_job(job_type=ElasticConstantsProject.job_type.Vasp, job_name=JobName)
        RelaxationJob.server.queue = "small"
        RelaxationJob.server.cores = 4
        
        RelaxationJob.structure = ActiveStructure.copy()
        
        RelaxationJob.input.incar["ENCUT"]  = 650
        RelaxationJob.input.incar["IBRION"] = -1
        RelaxationJob.input.incar["ISIF"]   = 2
        RelaxationJob.input.incar["NSW"]    = 0

        RelaxationJob.executable = VaspExecutable
        RelaxationJob.run()
    WaitForJobs(ElasticConstantsProject)

In [ ]:
def CalculateBulkModulus(Structure):
    StrainMagnitudes_Percent = CreateStrainRange(ZeroStrainPoint_Percent=0, Range_Percent=0.01, NumberOfStrains=11)
    Log.WriteInformation(f"Strains [%]: {StrainMagnitudes_Percent}")
    for Strain_Percent in StrainMagnitudes_Percent:
        StrainFactor     = 1 + Strain_Percent / 100
        TargetVolume_A3  = Structure.get_volume() * StrainFactor ** 3

        ActiveStructure, _ = SetVolume(Structure, TargetVolume_A3)

        JobName = f"BulkModulusCalculation_{ConvertToDirectoryName(round(Strain_Percent, 4))}"
        RelaxationJob = ElasticConstantsProject.create_job(job_type=ElasticConstantsProject.job_type.Vasp, job_name=JobName)
        RelaxationJob.server.queue = "small"
        RelaxationJob.server.cores = 4
        
        RelaxationJob.structure = ActiveStructure.copy()
        
        RelaxationJob.input.incar["ENCUT"]  = 650
        RelaxationJob.input.incar["IBRION"] = -1
        RelaxationJob.input.incar["ISIF"]   = 2
        RelaxationJob.input.incar["NSW"]    = 0

        RelaxationJob.executable = VaspExecutable
        RelaxationJob.run()
    WaitForJobs(ElasticConstantsProject)

In [ ]:
def PlotEnergyVolumeCurve(Log, Project, Name):
    Volumes_A3, Energies_eV = [], []
    for Job in Project.iter_jobs(convert_to_object=True):
        if not Job.job_name.startswith(Name) or not Job.status.finished:
            continue
        Volumes_A3.append(Job.output.volume[-1])
        Energies_eV.append(Job.output.energy_pot[-1])

    Order       = np.argsort(Volumes_A3)
    Volumes_A3  = np.array(Volumes_A3)[Order]
    TotalEnergies_eV = np.array(Energies_eV)[Order]
    MinimumEnergy_eV = np.min(TotalEnergies_eV)
    SetEquilibriumVolume_A3 = Volumes_A3[np.argmin(TotalEnergies_eV)]
    ElasticEnergies_eV = [(Entry-MinimumEnergy_eV)*1000 for Entry in TotalEnergies_eV]

    Log.WriteInformation(f"Volumes           [A^3]: [{', '.join(f'{Data:.4f}' for Data in Volumes_A3)}]")
    Log.WriteInformation(f"Energies          [ eV]: [{', '.join(f'{Data:.4f}' for Data in Energies_eV)}]")
    Log.WriteInformation(f"Elastic Energies  [meV]: [{', '.join(f'{Data:.4f}' for Data in ElasticEnergies_eV)}]")

    PolynomialFitCoefficients = np.polyfit(Volumes_A3, ElasticEnergies_eV, deg=3)
    FitX = np.linspace(min(Volumes_A3), max(Volumes_A3), 100)
    FitY = np.polyval(PolynomialFitCoefficients, FitX)
    RValue = np.corrcoef(ElasticEnergies_eV, np.polyval(PolynomialFitCoefficients, Volumes_A3))[0,1]
    BulkModulus_eV_A3_P3 = (2 * PolynomialFitCoefficients[1] + 6 * PolynomialFitCoefficients[0] * SetEquilibriumVolume_A3) * SetEquilibriumVolume_A3

    ASE_EquilibriumVolume_A3, ASE_EquilibriumEnergy_eV, BulkModulus_eV_A3_ASE = EquationOfState(Volumes_A3, ElasticEnergies_eV, eos='p3').fit()

    VolumeMismatch_Percent = 100 * np.abs(ASE_EquilibriumVolume_A3 - SetEquilibriumVolume_A3)/SetEquilibriumVolume_A3
    Log.WriteInformation(f"Zero Strain Volume = {round(SetEquilibriumVolume_A3, 3)} A^3")
    Log.WriteInformation(f"Calculated Volume  = {round(ASE_EquilibriumVolume_A3, 3)} A^3")
    Log.WriteInformation(f"Volume Mismatch    = {round(VolumeMismatch_Percent, 3)} %")

    plt.scatter(Volumes_A3, ElasticEnergies_eV, color="black", marker="o", label="VASP Data")
    plt.scatter([ASE_EquilibriumVolume_A3], [ASE_EquilibriumEnergy_eV], color="blue", marker="o", label="Calculated minimum")
    plt.plot(FitX, FitY, color="red", linestyle="--", label=f"3rd order polynomial fit")
    
    plt.xlabel(r"Volume ($\mathrm{\AA}^3$)")
    plt.ylabel("Elastic Energy (meV)")
    plt.title(f"E-V curve; $R²$ = {round(RValue, 4)}; Strain difference = {round(VolumeMismatch_Percent, 3)} %")
    
    plt.grid()
    plt.legend()
    plt.gca().ticklabel_format(useOffset=False, style="plain")
    plt.tight_layout()
    plt.savefig(f"Outputs/{Name}")
    plt.clf()

    BulkModulus_GPa_ASE = BulkModulus_eV_A3_ASE * ConversionFactor_eV_A3toGPa
    BulkModulus_GPa_P3 = BulkModulus_eV_A3_P3 * ConversionFactor_eV_A3toGPa
    BulkModulus_GPa = 0.5 * (BulkModulus_GPa_ASE + BulkModulus_GPa_P3)
    Error_GPa = np.abs(BulkModulus_GPa_ASE - BulkModulus_GPa_P3)
    if "BulkModulus" in Name:
        Log.WriteInformation(f"| Method     | Bulk Modulus [GPa] |")
        Log.WriteInformation(f"|------------|--------------------|")
        Log.WriteInformation(f"| ASE        | {BulkModulus_GPa_ASE:>18.4f} |")
        Log.WriteInformation(f"| Polynomial | {BulkModulus_GPa_P3:>18.4f} |")
        Log.WriteInformation(f"| Average    | {BulkModulus_GPa:>18.4f} |")
        Log.WriteInformation(f"|------------|--------------------|")
        Log.WriteInformation(f"Error: {Error_GPa} GPa")

    return ASE_EquilibriumVolume_A3, VolumeMismatch_Percent, BulkModulus_GPa

In [ ]:
Log.WriteInformation("Cell relaxation", WriteToNewParagraph=True, InsertTimeStamp=True)
MaximumIterations = 5
MismatchThreshold_Percent = 0.01

Log.WriteInformation(f"Energy minimization; Iteration 0", InsertTimeStamp=True)
Log.WriteInformation(f"Current lattice parameter: {RelaxedStructure.cell[0][0]} A")
RunUniaxialStrains(Log, 1, RelaxedStructure, 0)
ASE_EquilibriumVolume_A3, VolumeMismatch_Percent, _ = PlotEnergyVolumeCurve(Log, ElasticConstantsProject, f"StructureRelaxation_Iter0")
RelaxedStructure, RemainingStrain = SetVolume(RelaxedStructure, ASE_EquilibriumVolume_A3)
Log.WriteInformation(f"New lattice parameter: {RelaxedStructure.cell[0][0]} A")

ActiveIteration = 1
while VolumeMismatch_Percent > 1 + MismatchThreshold_Percent and ActiveIteration < MaximumIterations:
    Log.WriteInformation(f"Energy minimization; Iteration {ActiveIteration+1}", InsertTimeStamp=True)
    Log.WriteInformation(f"Current lattice parameter: {RelaxedStructure.cell[0][0]} A")
    RunUniaxialStrains(Log, 0.1, RelaxedStructure, ActiveIteration)
    ASE_EquilibriumVolume_A3, VolumeMismatch_Percent, _ = PlotEnergyVolumeCurve(Log, ElasticConstantsProject, f"StructureRelaxation_Iter{ActiveIteration}")
    RelaxedStructure, RemainingStrain = SetVolume(RelaxedStructure, ASE_EquilibriumVolume_A3)
    Log.WriteInformation(f"New lattice parameter: {RelaxedStructure.cell[0][0]} A")
    ActiveIteration += 1

In [ ]:
Log.WriteInformation("Bulk modulus calculation", WriteToNewParagraph=True, InsertTimeStamp=True)

CalculateBulkModulus(RelaxedStructure)
_, _, BulkModulus_GPa = PlotEnergyVolumeCurve(Log, ElasticConstantsProject, f"BulkModulusCalculation")
print(f"Bulk Modulus = {BulkModulus_GPa} GPa")